# Sección 0 — Exploración del dataset crudo
**Responsable:** Jesús · `src/limpieza.py`

**Entrada:** `data/raw/asteroids_data.csv`  

Este notebook es **únicamente exploratorio**. La limpieza y la exportación de los parquet
viven en `src/limpieza.py`. Para regenerar los archivos procesados ejecuta:
```
python -m src.limpieza
```

---

**Preguntas que guían la exploración:**
1. ¿Qué problemas de calidad tiene el dataset crudo?
2. ¿Cuántos eventos de aproximación reales contiene?
3. ¿Qué decisiones de limpieza justifican los datos?

In [ ]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

df_raw = pd.read_csv("../data/raw/asteroids_data.csv")
print(f"{df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

## 1. Estructura del dataset

In [ ]:
df_raw.dtypes

In [ ]:
df_raw.head(3)

## 2. Calidad de datos

In [ ]:
# Gráfica: porcentaje de nulos por columna
null_pct = (df_raw.isnull().mean() * 100).sort_values(ascending=False)
null_pct = null_pct[null_pct > 0]

fig, ax = plt.subplots(figsize=(8, 4))
null_pct.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_ylabel("% de valores nulos")
ax.set_xlabel("Columna")
ax.set_title("Porcentaje de valores nulos por columna")
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3, fontsize=9)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(null_pct.to_string())

In [ ]:
# Columna constante
print("Valores únicos en 'equinox':")
print(df_raw["equinox"].value_counts())

# Duplicados
print(f"\nIDs duplicados    : {df_raw['id'].duplicated().sum()}")
print(f"Nombres duplicados: {df_raw['name'].duplicated().sum()}")

**Hallazgos:**
- `short_name` tiene 91.6 % de nulos y `sentry_data` un 99.85 % → se descartan o rellenan.
- `equinox` es constante (`J2000`) → columna inútil, se descarta.
- Sin duplicados de ID ni nombre.
- Columnas `nasa_url` y `api_url` no aportan valor analítico → se descartan.
- `orbit_class_desc` y `orbit_class_range` son texto libre redundante con `orbit_class_type` → se descartan.

## 3. Exploración de `close_approach_data`

In [ ]:
# Estructura del primer evento de aproximación del primer asteroide
primer_evento = ast.literal_eval(df_raw["close_approach_data"].iloc[0])[0]
for k, v in primer_evento.items():
    print(f"{k}: {v}")

In [ ]:
# Número de eventos por asteroide
n_eventos = df_raw["close_approach_data"].apply(
    lambda x: len(ast.literal_eval(x))
)
print(f"Total eventos   : {n_eventos.sum():,}")
print(f"Promedio/ast.   : {n_eventos.mean():.1f}")
print(f"Mínimo / Máximo : {n_eventos.min()} / {n_eventos.max()}")
n_eventos.describe()

## 4. Cuerpos orbitados

In [ ]:
# Extraer únicamente el campo orbiting_body de cada evento (sin guardar)
cuerpos = [
    e["orbiting_body"]
    for row in df_raw["close_approach_data"]
    for e in ast.literal_eval(row)
]
por_cuerpo = pd.Series(cuerpos).value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
por_cuerpo.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Número de eventos de aproximación por cuerpo orbitado")
ax.set_ylabel("Número de eventos")
ax.set_xlabel("Cuerpo orbitado")
ax.bar_label(ax.containers[0], fmt="%d", padding=3, fontsize=9)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

print(por_cuerpo.to_string())

## Hallazgos y decisiones de limpieza

| Hallazgo | Decisión en `src/limpieza.py` |
|---|---|
| `equinox` constante | Descartar |
| `nasa_url`, `api_url` sin valor analítico | Descartar |
| `sentry_data` 99.85 % nulo | Descartar; `sentry` booleano lo cubre |
| `short_name` 91.6 % nulo | Rellenar con `""` |
| `orbit_class_desc/range` texto libre redundante | Descartar |
| Fechas como strings | Convertir a `datetime64` |
| `close_approach_data` anidada | Aplanar en tabla independiente |
| ~64 000 eventos en 6 cuerpos (Tierra 65 %) | Tabla `approaches_clean` con 9 columnas |

La implementación completa está en **`src/limpieza.py`**.